# Поиск услуг: кандидатогенерация

До 50 объявлений на запрос. Основная метрика — macro Recall@50.

## 1. Настройки

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import os
import importlib.metadata
import platform
import random
import shutil
import subprocess
from time import perf_counter

import bm25s
import torch
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
from IPython.display import display

DATA_DIR = Path("dataset")
SEED = 42
DEVICE_REQUEST = os.environ.get("AVITO_DEVICE", "auto").lower()
VALIDATION_SIZE = 0.2
SPLIT_PATH = Path("split.csv")
EXPERIMENTS_PATH = Path("experiments.csv")
BM25_CONFIG = {"k1": 1.2, "b": 0.75, "method": "lucene", "backend": "numpy"}
TOKEN_PATTERN = r"(?u)\b\w+\b"
RECALL_K = (1, 5, 10, 20, 50)
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

In [2]:
def select_device(requested: str) -> torch.device:
    if requested not in {"auto", "cpu", "cuda", "mps"}:
        raise ValueError("AVITO_DEVICE: auto, cpu, cuda или mps")
    if requested == "auto":
        requested = ("cuda" if torch.cuda.is_available() else
                     "mps" if torch.backends.mps.is_available() else "cpu")
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна: проверьте GPU, драйвер и сборку PyTorch")
    if requested == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS недоступен: требуется совместимый Mac и сборка PyTorch")
    return torch.device(requested)


def check_device(device: torch.device) -> None:
    # Проверяем вычисления и градиенты, а не только наличие устройства.
    x = torch.arange(12, dtype=torch.float32).reshape(3, 4).to(device)
    x.requires_grad_(True)
    loss = (x @ x.T).sum()
    loss.backward()
    expected_gradient = 2 * torch.arange(12, dtype=torch.float32).reshape(3, 4).sum(0)
    torch.testing.assert_close(x.grad.cpu(), expected_gradient.expand(3, 4))
    torch.testing.assert_close(loss.detach().cpu(), torch.tensor(1134.0))


DEVICE = select_device(DEVICE_REQUEST)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
elif DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)
check_device(DEVICE)
print(f"PyTorch {torch.__version__}; устройство: {DEVICE}; вычисления и градиенты: OK")

PyTorch 2.14.0+cu130; устройство: cpu; вычисления и градиенты: OK


## 2. Данные

In [3]:
QUERY_FIELDS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]


def normalize_text(values: pd.Series) -> pd.Series:
    """Нижний регистр и нормализация пробелов."""
    return values.fillna("").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resources(root: Path) -> dict:
    try:
        gpu = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, check=True, timeout=10,
        ).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        gpu = "unavailable"
    return {
        "python": platform.python_version(), "platform": platform.platform(),
        "cpu_physical": psutil.cpu_count(logical=False), "cpu_logical": psutil.cpu_count(),
        "ram_total_bytes": psutil.virtual_memory().total,
        "ram_available_bytes": psutil.virtual_memory().available,
        "disk_free_bytes": shutil.disk_usage(root).free, "gpu": gpu,
        "versions": {name: importlib.metadata.version(name) for name in
                     ["pandas", "numpy", "pyarrow", "nbformat", "nbclient", "psutil", "bm25s", "scikit-learn", "torch"]},
    }

In [4]:
def audit_data(data_dir: Path) -> dict:
    tables, files, columns, examples = {}, [], [], {}
    for name in ["benchmark_items", "benchmark_queries", "train"]:
        path = data_dir / f"{name}.parquet"
        frame = pd.read_parquet(path)
        tables[name] = frame
        files.append({"file": path.name, "bytes": path.stat().st_size,
                      "rows": len(frame), "columns": len(frame.columns), "sha256": sha256(path)})
        schema = pq.read_schema(path)
        for column in frame:
            values = frame[column]
            columns.append({"table": name, "column": column, "type": str(schema.field(column).type),
                            "nulls": int(values.isna().sum()), "unique": int(values.nunique()),
                            "empty_strings": int(values.eq("").sum())})
        examples[name] = frame.head(3).map(
            lambda x: str(x)[:240] if pd.notna(x) else None)

    items, queries, train = (tables[n] for n in ["benchmark_items", "benchmark_queries", "train"])
    for frame, key, pattern in [(items, "item_id", r"[0-9a-f]{16}"),
                                (train, "item_id", r"[0-9a-f]{16}"),
                                (queries, "query_id", r".{16}")]:
        if not frame[key].str.fullmatch(pattern).fillna(False).all():
            raise ValueError(f"Invalid {key}")
    if not items.item_id.is_unique or not queries.query_id.is_unique:
        raise ValueError("Duplicate benchmark IDs require an explicit resolution")

    # В train нет query_id; группируем по тексту и контексту.
    identity = train[QUERY_FIELDS].copy()
    identity["search_query"] = normalize_text(identity.search_query)
    group_ids = identity.groupby(QUERY_FIELDS, dropna=False, sort=True).ngroup()
    pairs = pd.DataFrame({"group": group_ids, "item_id": train.item_id}).drop_duplicates()
    pairs["in_corpus"] = pairs.item_id.isin(items.item_id)
    positives = pairs.groupby("group").agg(positives=("item_id", "size"), in_corpus=("in_corpus", "sum"))
    contexts = identity.drop_duplicates().groupby("search_query", dropna=False).size()
    item_text = pd.DataFrame({field: normalize_text(items[field]) for field in
                              ["item_title_raw", "item_description_raw"]})
    title_counts = item_text.groupby("item_title_raw").size()
    summary = {
        "train_rows": len(train), "corpus_items": len(items), "benchmark_queries": len(queries),
        "train_has_query_id": "query_id" in train, "train_has_explicit_label": False,
        "label_assumption": "Each train row is a positive interaction; no explicit label is provided.",
        "query_context_groups": len(positives),
        "normalized_train_texts": int(identity.search_query.nunique()),
        "texts_with_multiple_contexts": int(contexts.gt(1).sum()),
        "duplicate_positive_rows_after_normalization": len(train) - len(pairs),
        "unique_positive_pairs": len(pairs),
        "positive_pairs_outside_corpus": int((~pairs.in_corpus).sum()),
        "unique_positive_items_outside_corpus": int(pairs.loc[~pairs.in_corpus, "item_id"].nunique()),
        "groups_without_retrievable_positive": int(positives.in_corpus.eq(0).sum()),
        "positive_count_distribution": positives.positives.describe(percentiles=[.5, .9, .99]).to_dict(),
        "macro_recall_corpus_ceiling": float((positives.in_corpus / positives.positives).mean()),
        "empty_train_query_rows": int(identity.search_query.eq("").sum()),
        "empty_benchmark_query_rows": int(normalize_text(queries.search_query).eq("").sum()),
        "benchmark_repeated_text_rows": int(normalize_text(queries.search_query).duplicated().sum()),
        "duplicate_item_text_rows": int(item_text.duplicated().sum()),
        "repeated_title_groups": int(title_counts.gt(1).sum()),
        "near_duplicate_status": "Not measured; repeated titles alone do not prove near duplication.",
        "event_time_columns": [c for c in train if any(t in c.lower() for t in ["timestamp", "date", "event_time"])],
        "dense_float32_384_bytes": len(items) * 384 * 4,
        "dense_float32_768_bytes": len(items) * 768 * 4,
    }
    return {
        "summary": summary, "files": pd.DataFrame(files),
        "columns": pd.DataFrame(columns), "examples": examples,
        "resources": resources(data_dir.parent),
    }

In [5]:
audit = audit_data(DATA_DIR)
summary = audit["summary"]
files = audit["files"]
files["Размер, МиБ"] = (files["bytes"] / 2**20).round(2)
display(files[["file", "rows", "columns", "Размер, МиБ"]].rename(
    columns={"file": "Файл", "rows": "Строки", "columns": "Поля"}))
display(files[["file", "sha256"]])

,Файл,Строки,Поля,"Размер, МиБ"
0,benchmark_items.parquet,189212,14,186.61
1,benchmark_queries.parquet,2452,6,0.12
2,train.parquet,497673,19,467.53


,file,sha256
0,benchmark_items.parquet,193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899
1,benchmark_queries.parquet,e49de4fb76f03979a4af96034817767d39ae5de9f94e254fed028243188dda06
2,train.parquet,e150ab7a5c98769f643b95ee3a02dc0f664b647facd69a7d280ec6d48ca554a7


`benchmark_items` — корпус объявлений, `benchmark_queries` — итоговые запросы,
`train` — пары запрос–объявление.

In [6]:
columns = audit["columns"]
column_names = {"column": "Поле", "type": "Тип Parquet", "nulls": "Пропуски",
                "unique": "Уникальные значения", "empty_strings": "Пустые строки"}
for table in files["file"].str.removesuffix(".parquet"):
    print(table)
    display(columns.loc[columns["table"].eq(table)].drop(columns="table").rename(columns=column_names))

benchmark_items


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
0,item_title_raw,string,0,119952,0
1,item_rating_reviews_count,double,11615,1170,0
2,item_rating,double,17631,6007,0
3,item_price,"decimal128(27, 15)",0,2949,0
4,item_microcat_id,int64,0,752,0
5,item_longitude,"decimal128(18, 15)",1,120004,0
6,item_location_id,int64,0,2877,0
7,item_latitude,"decimal128(17, 15)",1,119778,0
8,item_is_phone_hidden,bool,0,2,0
9,item_is_message_forbidden,bool,0,2,0


benchmark_queries


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
14,query_id,large_string,0,2452,0
15,search_query,large_string,0,2452,0
16,search_location_id,int64,0,273,0
17,search_is_delivery_search,int32,0,1,0
18,search_infm_params_text,large_string,0,132,1546
19,search_category,int64,0,2,0


train


,Поле,Тип Parquet,Пропуски,Уникальные значения,Пустые строки
20,search_query,string,0,74529,0
21,search_location_id,int64,0,2782,0
22,search_is_delivery_search,int32,0,2,0
23,search_infm_params_text,string,0,3724,163999
24,search_category,int64,0,4,0
25,item_title_raw,string,0,210029,0
26,item_rating_reviews_count,double,19211,1372,0
27,item_rating,double,28232,8832,0
28,item_price,"decimal128(27, 15)",0,2992,0
29,item_microcat_id,int64,0,212,0


### Примеры

Для отображения обрезаем значения до 240 символов.

In [7]:
for table, title in [("benchmark_queries", "Итоговый запрос"),
                     ("benchmark_items", "Объявление из корпуса"),
                     ("train", "Строка обучающей разметки")]:
    examples = audit["examples"][table]
    print(title)
    display(examples.iloc[0].rename("Пример значения").to_frame())

Итоговый запрос


,Пример значения
query_id,70DfDUpwjxB4lzFd
search_query,перевозки владикавказ тбилиси
search_location_id,649820
search_is_delivery_search,0
search_infm_params_text,
search_category,114


Объявление из корпуса


,Пример значения
item_title_raw,Ремонт/выкуп компьют. и ноутбуков с выездом на дом
item_rating_reviews_count,57.0
item_rating,5.0
item_price,500.000000000000000
item_microcat_id,2097573
item_longitude,42.047193961004901
item_location_id,631060
item_latitude,44.228180450924000
item_is_phone_hidden,False
item_is_message_forbidden,False


Строка обучающей разметки


,Пример значения
search_query,скупка телевизоров
search_location_id,652430
search_is_delivery_search,0
search_infm_params_text,
search_category,114
item_title_raw,Скупка б/у техники
item_rating_reviews_count,91.0
item_rating,4.989010989010989
item_price,1.000000000000000
item_microcat_id,2303374


В train нет `query_id`, явной метки релевантности и времени событий.
Каждую строку считаем позитивным взаимодействием.
Для BM25 используем заголовок и описание.

## 3. Запросы и повторы

Запрос определяем сочетанием пяти `search_*` полей. В тексте приводим буквы к нижнему
регистру и нормализуем пробелы. Одинаковый текст с разными фильтрами — разные запросы
для метрики, но одна группа для split.

In [8]:
query_counts = {
    "Строки train": summary["train_rows"],
    "Различные нормализованные тексты": summary["normalized_train_texts"],
    "Запросы с учётом контекста": summary["query_context_groups"],
    "Тексты с несколькими контекстами": summary["texts_with_multiple_contexts"],
    "Повторные позитивные пары": summary["duplicate_positive_rows_after_normalization"],
    "Уникальные позитивные пары": summary["unique_positive_pairs"],
}
display(pd.Series(query_counts, name="Количество").to_frame())
distribution = summary["positive_count_distribution"]
display(pd.Series({"Минимум": distribution["min"], "Медиана": distribution["50%"],
                   "90-й перцентиль": distribution["90%"], "99-й перцентиль": distribution["99%"],
                   "Максимум": distribution["max"], "Среднее": distribution["mean"]},
                  name="Позитивов на запрос с контекстом").to_frame().round(2))

,Количество
Строки train,497673
Различные нормализованные тексты,73904
Запросы с учётом контекста,354313
Тексты с несколькими контекстами,27894
Повторные позитивные пары,30644
Уникальные позитивные пары,467029


,Позитивов на запрос с контекстом
Минимум,1.00
Медиана,1.00
90-й перцентиль,2.00
99-й перцентиль,6.00
Максимум,278.00
Среднее,1.32


У большинства запросов один позитив, у некоторых — сотни.
Повторные пары запрос–объявление учитываем один раз.

## 4. Покрытие позитивов корпусом

Проверяем наличие размеченных `item_id` в корпусе. Позитивы вне корпуса
оставляем в знаменателе Recall.

In [9]:
total = summary["unique_positive_pairs"]
outside = summary["positive_pairs_outside_corpus"]
coverage = pd.DataFrame({
    "Позитивные пары": [total - outside, outside],
}, index=["Объявление есть в корпусе", "Объявления нет в корпусе"])
coverage["Доля пар, %"] = (100 * coverage["Позитивные пары"] / total).round(2)
display(coverage)
display(pd.Series({
    "Запросы с контекстом": summary["query_context_groups"],
    "Из них без доступных позитивов": summary["groups_without_retrievable_positive"],
}, name="Количество").to_frame())
print(f"Верхняя граница macro Recall по доступности корпуса: {summary['macro_recall_corpus_ceiling']:.2%}")

,Позитивные пары,"Доля пар, %"
Объявление есть в корпусе,29571,6.33
Объявления нет в корпусе,437458,93.67


,Количество
Запросы с контекстом,354313
Из них без доступных позитивов,327764


Верхняя граница macro Recall по доступности корпуса: 6.08%


Вне корпуса — 437 458 из 467 029 позитивных пар.
Верхняя граница macro Recall на всех train-группах — **6,08%**;
ограничение top-50 может снизить её ещё сильнее.

## 5. Дубли объявлений и пустые запросы

Сравниваем нормализованные заголовки и описания.

In [10]:
display(pd.Series({
    "Повторные строки title + description в корпусе": summary["duplicate_item_text_rows"],
    "Группы повторяющихся заголовков": summary["repeated_title_groups"],
    "Пустые запросы в train": summary["empty_train_query_rows"],
    "Пустые итоговые запросы": summary["empty_benchmark_query_rows"],
    "Повторные тексты итоговых запросов": summary["benchmark_repeated_text_rows"],
}, name="Количество").to_frame())

,Количество
Повторные строки title + description в корпусе,19097
Группы повторяющихся заголовков,15195
Пустые запросы в train,0
Пустые итоговые запросы,0
Повторные тексты итоговых запросов,1


Найдено 19 097 повторов заголовка и описания при разных item_id.
Сохраняем все ID: метрика различает такие объявления.
Benchmark ID уникальны, пустых поисковых текстов нет.

## 6. Ресурсы

In [11]:
resource_info = audit["resources"]
display(pd.Series({
    "Python": resource_info["python"],
    "CPU: физические ядра / потоки": f"{resource_info['cpu_physical']} / {resource_info['cpu_logical']}",
    "RAM всего, ГиБ": round(resource_info["ram_total_bytes"] / 2**30, 1),
    "RAM доступно, ГиБ": round(resource_info["ram_available_bytes"] / 2**30, 1),
    "Диск свободно, ГиБ": round(resource_info["disk_free_bytes"] / 2**30, 1),
    "GPU / память, МиБ / драйвер": resource_info["gpu"],
}, name="Значение").to_frame())
display(pd.DataFrame({
    "Размерность": [384, 768],
    "Embeddings float32, МиБ": [round(summary[f"dense_float32_{d}_bytes"] / 2**20, 1) for d in [384, 768]],
}))
display(pd.Series(resource_info["versions"], name="Версия библиотеки").to_frame())

,Значение
Python,3.12.3
CPU: физические ядра / потоки,16 / 32
"RAM всего, ГиБ",60.4
"RAM доступно, ГиБ",44.7
"Диск свободно, ГиБ",1308.5
"GPU / память, МиБ / драйвер","NVIDIA GeForce RTX 5090, 32607 MiB, 595.91.07"


,Размерность,"Embeddings float32, МиБ"
0,384,277.2
1,768,554.3


,Версия библиотеки
pandas,3.0.6
numpy,2.5.3
pyarrow,25.0.1
nbformat,5.11.1
nbclient,0.11.0
psutil,7.2.2
bm25s,0.3.11
scikit-learn,1.9.1
torch,2.14.0


Embeddings float32 займут 277–554 МиБ при размерности 384–768,
без учёта модели, батчей и индекса.

## 7. Валидация

Откладываем 20% уникальных нормализованных текстов, seed 42.
Все контексты и позитивы одного текста остаются вместе. Повторные пары удаляем.
Временной split невозможен: в данных нет времени событий.

Словарь BM25 строим по корпусу объявлений. Validation labels используются только для оценки.

In [12]:
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet").sort_values("item_id").reset_index(drop=True)
train = pd.read_parquet(DATA_DIR / "train.parquet", columns=QUERY_FIELDS + ["item_id"])
train["search_query"] = normalize_text(train["search_query"])
texts = np.sort(train["search_query"].unique())
train_texts, validation_texts = train_test_split(
    texts, test_size=VALIDATION_SIZE, random_state=SEED, shuffle=True)
validation_texts = set(validation_texts)
text_ids = {text: hashlib.sha256(text.encode("utf-8")).hexdigest() for text in texts}
split = pd.DataFrame({
    "text_id": [text_ids[text] for text in texts],
    "split": ["validation" if text in validation_texts else "train" for text in texts],
}).sort_values("text_id").reset_index(drop=True)
assert split["text_id"].is_unique
if SPLIT_PATH.exists():
    pd.testing.assert_frame_equal(pd.read_csv(SPLIT_PATH), split)
else:
    split.to_csv(SPLIT_PATH, index=False)
SPLIT_HASH = sha256(SPLIT_PATH)

train["group_id"] = train.groupby(QUERY_FIELDS, dropna=False, sort=True).ngroup()
train["split"] = np.where(train["search_query"].isin(validation_texts), "validation", "train")
assert train.groupby("group_id")["split"].nunique().eq(1).all()
assert set(train_texts).isdisjoint(validation_texts)
assert len(train_texts) + len(validation_texts) == len(texts)
pairs = train[["group_id", "item_id"]].drop_duplicates()
queries = train.drop_duplicates("group_id").set_index("group_id")[["search_query", "split"]]
queries["positives"] = pairs.groupby("group_id")["item_id"].agg(frozenset)
corpus_ids = frozenset(items["item_id"])
queries["available"] = queries["positives"].map(lambda values: values & corpus_ids)
queries["n_positives"] = queries["positives"].map(len)
queries["n_available"] = queries["available"].map(len)
validation = queries.loc[queries["split"].eq("validation")].copy()
assert queries["n_positives"].gt(0).all()
assert len(pairs) == summary["unique_positive_pairs"]
display(queries.groupby("split").agg(
    texts=("search_query", "nunique"), groups=("search_query", "size"),
    positive_pairs=("n_positives", "sum"), available_pairs=("n_available", "sum")))
print(f"SHA-256 split: {SPLIT_HASH}")
print(f"Validation: {validation.n_available.eq(0).sum():,} групп без доступных позитивов")

,texts,groups,positive_pairs,available_pairs
split,,,,
train,59123,285932,379842,23844
validation,14781,68381,87187,5727


SHA-256 split: bc175ad59e9f1cd311579a22c1c23682aa27fbac75a03c6f6141fe4ee2e0732e
Validation: 63,229 групп без доступных позитивов


### Пересечение объявлений

Одно объявление может быть позитивом разных запросов в обеих частях split.
Считаем общие item_id и совпадения текстов среди доступных позитивов.
Объявления с одинаковыми текстами сохраняем под исходными ID.

In [13]:
train_positive_ids = frozenset(pairs.loc[pairs.group_id.isin(queries.index[queries.split.eq("train")]), "item_id"])
val_positive_ids = frozenset(pairs.loc[pairs.group_id.isin(validation.index), "item_id"])
item_signatures = pd.Series(list(zip(
    normalize_text(items["item_title_raw"]), normalize_text(items["item_description_raw"])
)), index=items.item_id)
train_signatures = set(item_signatures.loc[item_signatures.index.isin(train_positive_ids)])
val_corpus_signatures = item_signatures.loc[item_signatures.index.isin(val_positive_ids)]
display(pd.Series({
    "Общие позитивные item_id между частями": len(train_positive_ids & val_positive_ids),
    "Validation item_id в корпусе": len(val_corpus_signatures),
    "Из них с текстом среди train-позитивов корпуса": int(val_corpus_signatures.isin(train_signatures).sum()),
}, name="Количество").to_frame())
del item_signatures, train_signatures, val_corpus_signatures

,Количество
Общие позитивные item_id между частями,25513
Validation item_id в корпусе,4809
Из них с текстом среди train-позитивов корпуса,2269


## 8. Метрики

Macro Recall@K усредняем по запросам с контекстом. Позитивы вне корпуса остаются
в знаменателе. Дубликаты выдачи удаляем до отсечения top-K.
Запросы без разметки исключаем из среднего и считаем отдельно.

Дополнительно считаем HitRate@50 и Recall@50 по доступным позитивам —
только среди групп, где есть хотя бы один такой позитив.

In [14]:
def query_metrics(positives, candidates, ks=RECALL_K):
    positives = set(positives)
    unique = list(dict.fromkeys(candidates))
    recalls = {f"Recall@{k}": len(positives.intersection(unique[:k])) / len(positives)
               if positives else np.nan for k in ks}
    recalls["HitRate@50"] = float(bool(positives.intersection(unique[:50]))) if positives else np.nan
    return recalls


def evaluate(query_table, predictions):
    rows = []
    for query in query_table.itertuples():
        candidates = predictions[query.search_query]
        values = query_metrics(query.positives, candidates)
        values["available_Recall@50"] = query_metrics(query.available, candidates)["Recall@50"]
        rows.append(values)
    return pd.DataFrame(rows, index=query_table.index)


assert query_metrics({"a", "b"}, ["a", "a", "b"], (1, 2))["Recall@2"] == 1
assert query_metrics({"a", "outside"}, ["a"])["Recall@50"] == 0.5
assert query_metrics({"outside"}, ["a"])["Recall@50"] == 0
assert query_metrics({"a"}, [])["HitRate@50"] == 0
assert np.isnan(query_metrics(set(), ["a"])["Recall@50"])
# Macro: равный вес запросов, независимо от числа позитивов.
assert np.mean([query_metrics({"a", "b"}, ["a"])["Recall@50"],
                query_metrics({"c"}, ["c"])["Recall@50"]]) == 0.75
print("Метрики: проверки пройдены")

Метрики: проверки пройдены


## 9. E0 — BM25

BM25 по заголовку и описанию: `bm25s`, вариант `lucene`, `k1=1.2`, `b=0.75`, float32.
Параметры без подбора.

Нижний регистр, нормализация пробелов, токены `\w+`. Сохраняем одиночные буквы,
числа, стоп-слова и «ё». Без стемминга и лемматизации.

При равных оценках сортируем по item_id. Нулевые оценки не включаем.
Один текст ищем один раз, каждый контекст оцениваем отдельно.

In [15]:
def tokenize(texts):
    return bm25s.tokenize(texts, lower=False, stopwords=[], token_pattern=TOKEN_PATTERN,
                          return_ids=False, show_progress=False)


def bm25_scores(model, tokens):
    # get_scores() в bm25s не принимает пустой список токенов.
    return model.get_scores_from_ids(model.get_tokens_ids(tokens))


def top_k(scores, k=50):
    if k <= 0:
        return np.empty(0, dtype=np.int64)
    selected = np.flatnonzero(scores > 0)
    if len(selected) > k:
        threshold = np.partition(scores[selected], -k)[-k]
        higher = selected[scores[selected] > threshold]
        tied = selected[scores[selected] == threshold][:k - len(higher)]
        selected = np.concatenate([higher, tied])
    return selected[np.lexsort((selected, -scores[selected]))]


assert tokenize(["ё  Ё  A-15 и"])[0] == ["ё", "Ё", "A", "15", "и"]
assert top_k(np.array([2., 1., 1., 0.]), 2).tolist() == [0, 1]
assert top_k(np.zeros(3)).tolist() == []
assert top_k(np.ones(60), 50).tolist() == list(range(50))
rng = np.random.default_rng(SEED)
for size in (1, 20, 100):
    scores = rng.integers(0, 5, size).astype(np.float32)
    expected = sorted(np.flatnonzero(scores > 0), key=lambda i: (-scores[i], i))[:50]
    assert top_k(scores).tolist() == expected

small_ids = np.array(["0000000000000001", "0000000000000002", "0000000000000003"])
small = bm25s.BM25(**BM25_CONFIG)
small.index(tokenize(["ремонт авто", "ремонт авто", "уборка кухни"]), show_progress=False)
assert small_ids[top_k(bm25_scores(small, ["ремонт"]))].tolist() == small_ids[:2].tolist()
assert top_k(bm25_scores(small, ["несуществующийтокен"])).tolist() == []
assert top_k(bm25_scores(small, [])).tolist() == []
# Для одинаковых документов длины 2 оценка каждого совпадения известна из формулы.
expected_score = np.log(1 + (3 - 2 + 0.5) / (2 + 0.5)) / (1 + BM25_CONFIG["k1"])
np.testing.assert_allclose(bm25_scores(small, ["ремонт"])[:2], expected_score, rtol=1e-6)
print("BM25, top-k и соответствие item_id: проверки пройдены")

BM25, top-k и соответствие item_id: проверки пройдены


In [16]:
start = perf_counter()
corpus_text = normalize_text(items["item_title_raw"].fillna("") + " " + items["item_description_raw"].fillna(""))
corpus_tokens = tokenize(corpus_text.tolist())
retriever = bm25s.BM25(**BM25_CONFIG)
retriever.index(corpus_tokens, show_progress=False)
index_seconds = perf_counter() - start
item_ids = items["item_id"].to_numpy()
assert items["item_id"].is_unique and items["item_id"].is_monotonic_increasing
assert retriever.scores["num_docs"] == len(item_ids)
index_bytes = sum(value.nbytes for value in retriever.scores.values() if isinstance(value, np.ndarray))
del corpus_tokens
_ = gc.collect()
print(f"Индексация: {index_seconds:.2f} с; массивы индекса: {index_bytes / 2**20:.1f} МиБ")

Индексация: 21.09 с; массивы индекса: 183.3 МиБ


In [17]:
predictions, query_seconds = {}, []
val_texts = sorted(validation["search_query"].unique())
start = perf_counter()
val_tokens = tokenize(val_texts)
for text, tokens in zip(val_texts, val_tokens):
    query_start = perf_counter()
    positions = top_k(bm25_scores(retriever, tokens), max(RECALL_K))
    predictions[text] = item_ids[positions].tolist()
    query_seconds.append(perf_counter() - query_start)
search_seconds = perf_counter() - start
assert set(predictions) == set(val_texts)
assert all(len(ids) <= 50 and len(ids) == len(set(ids)) and set(ids) <= corpus_ids
           for ids in predictions.values())
per_query = evaluate(validation, predictions)
metrics = per_query.mean()
assert per_query["Recall@50"].notna().all()
assert per_query[[f"Recall@{k}" for k in RECALL_K]].diff(axis=1).iloc[:, 1:].ge(0).all().all()
ceiling = (validation["n_available"].clip(upper=50) / validation["n_positives"]).mean()
assert metrics["Recall@50"] <= ceiling + 1e-12
display(metrics.rename("Значение").to_frame().round(6))
display(pd.Series({
    "Validation-группы": len(validation),
    "Группы без разметки": int(validation.n_positives.eq(0).sum()),
    "Группы с доступными позитивами": int(validation.n_available.gt(0).sum()),
    "Уникальные тексты для поиска": len(val_texts),
    "Тексты без кандидатов": sum(not values for values in predictions.values()),
    "Верхняя граница Recall@50": ceiling,
    "Индексация, с": index_seconds,
    "Поиск с токенизацией, с": search_seconds,
    "Поиск без токенизации, p50 мс": float(np.percentile(query_seconds, 50) * 1000),
    "Поиск без токенизации, p95 мс": float(np.percentile(query_seconds, 95) * 1000),
    "Массивы индекса, МиБ": index_bytes / 2**20,
    "RSS процесса после поиска, МиБ": psutil.Process().memory_info().rss / 2**20,
}, name="Значение").to_frame())

,Значение
Recall@1,0.001626
Recall@5,0.005165
Recall@10,0.008089
Recall@20,0.012069
Recall@50,0.019168
HitRate@50,0.023369
available_Recall@50,0.297620


,Значение
Validation-группы,68381.000000
Группы без разметки,0.000000
Группы с доступными позитивами,5152.000000
Уникальные тексты для поиска,14781.000000
Тексты без кандидатов,45.000000
Верхняя граница Recall@50,0.061306
"Индексация, с",21.086560
"Поиск с токенизацией, с",3.490715
"Поиск без токенизации, p50 мс",0.174952
"Поиск без токенизации, p95 мс",0.546558


Поиск последовательный, на CPU. Индексация включает подготовку текста и токенизацию.
RSS показывает память всего процесса после поиска, а не пиковое потребление индекса.

In [18]:
experiment = {
    "experiment": "E0", "status": "completed", "config": json.dumps(BM25_CONFIG, sort_keys=True),
    "token_pattern": TOKEN_PATTERN, "fields": "title + description", "seed": SEED,
    "validation_size": VALIDATION_SIZE, "split_sha256": SPLIT_HASH,
    "corpus_sha256": sha256(DATA_DIR / "benchmark_items.parquet"),
    "train_sha256": sha256(DATA_DIR / "train.parquet"),
    "bm25s_version": importlib.metadata.version("bm25s"), "device": "cpu",
    **{name: float(value) for name, value in metrics.items()}, "delta_Recall@50": 0.0,
    "index_seconds": index_seconds, "search_seconds": search_seconds,
    "index_MiB": index_bytes / 2**20, "validation_groups": len(validation),
    "conclusion": f"Recall@50={metrics['Recall@50']:.4%}; по доступным позитивам={metrics['available_Recall@50']:.2%}",
}
experiments = pd.DataFrame([experiment])
# Повторный запуск обновляет E0; результаты других экспериментов сохраняются.
if EXPERIMENTS_PATH.exists():
    previous = pd.read_csv(EXPERIMENTS_PATH)
    experiments = pd.concat([previous.loc[previous.experiment.ne("E0")], experiments], ignore_index=True)
experiments.to_csv(EXPERIMENTS_PATH, index=False)
display(experiments[["experiment", "status", "Recall@1", "Recall@5", "Recall@10", "Recall@20", "Recall@50"]])

,experiment,status,Recall@1,Recall@5,Recall@10,Recall@20,Recall@50
0,E0,completed,0.001626,0.005165,0.008089,0.012069,0.019168


BM25: **Recall@50 = 1,92%** по 68 381 validation-группе.
У 63 229 групп нет позитивов в корпусе; верхняя граница Recall@50 — **6,13%**.
По доступным позитивам среди 5 152 групп Recall@50 — **29,76%**.
У 45 текстов нет совпадений со словарём корпуса.